<a href="https://colab.research.google.com/github/pavankumarcode/Mastering-AI/blob/main/3__Agent__The_Autonomous_IT_Support_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment: The Autonomous IT Support Agent**

Build and Agent for the **Level 1 IT Incident Responder**.

**Objective:** You are building an AI agent that acts as the "first responder" for server incidents. It must:

1. **Investigate:** Check server health and logs when a user reports an issue.
2. **Act:** If CPU is critical (>90%), it should **Restart** the service.
3. **Escalate:** If the issue is complex or logs show "Payment Gateway Error", it should **Escalate** to a human.

# Step 1 - Initialize the Agent - Get all **Imports**

In [ ]:
import os
import json
from openai import OpenAI
from google.colab import userdata

## Set up the connection to OpenAI

In [ ]:
# 1. Initialize OpenAI Client
try:
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
except Exception as e:
    print(f"Error initializing OpenAI client: [{e}]")
    print("Please ensure your OPENAI_API_KEY is set in your environment variables.")

## Set the Debugging Option

In [ ]:
DEBUG = False

# Step 2 - Set up the Tools

## Tool 1 - Fucntion to Check the Server Health

In [2]:
"""
Fucntion that returns CPU and Memory usage for a given server.
"""
def get_server_health(server_id: str) -> str:

    print(f"Tool Called - get_server_health : Checking the Server Health for Server [{server_id}].")

    dummy_data = {
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},  # Scenario 1: High CPU (Needs Restart)
        "db-node-02"       : {"cpu": "12%", "memory": "60%", "status": "Healthy"},  # Scenario 2: Healthy (No Action Needed)
        "auth-service-03"  : {"cpu": "45%", "memory": "95%", "status": "Critical"}, # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "search-index-09"  : {"cpu": "10%", "memory": "15%", "status": "Error"},    # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "frontend-node-04" : {"cpu": "25%", "memory": "30%", "status": "Healthy"},  # Scenario 5: Completely Normal
    }

    result = dummy_data.get(server_id, {"error": "Server not found. Check the ID."})
    json_result = json.dumps(result)

    if DEBUG:
      print(f'DEBUG - Tool get_server_health - Server [{server_id}], Health [{json_result}]')

    return json_result


## Tool 2 - Fucntion to get the logs for a specific Server

In [3]:
"""
Fucntion that returns last N lines of logs for the given Server
"""
def get_recent_logs(server_id: str, lines: int = 5) -> str:

    print(f"Tool Called - get_recent_logs : Getting last [{lines}] lines of Logs for [{server_id}].")

    # Different logs for different servers to trigger different agent behaviors
    dummy_log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = dummy_log_database.get(server_id, default_logs)

    json_logs = json.dumps({"logs": logs[:lines]})

    if DEBUG:
      print(f'DEBUG - Tool get_recent_logs - Server [{server_id}], Health [{json_logs}]')

    return json_logs

## Tool 3 - Function to Restart a Given Server

In [1]:
"""
Fucntion that Restarts the Given Server
"""
def restart_server(server_id: str) -> str:

    print(f"Tool Called - restart_server : Restarting the Server [{server_id}].")

    # In a real scenario, this would run a subprocess command or API call
    result = {
        "server_id": server_id,
        "status": "success",
        "message": "Service restart command issued successfully."
    }

    result = json.dumps(result)

    if DEBUG:
      print(f'DEBUG - Tool restart_server - Server [{server_id}], Restart Complted - Logs [{result}]')

    return result

## Tool 4 - Esclate to Engineer

In [ ]:
"""
Fucntion That sends Esclation to Engineer.
Alternatively we can create a P1 Incident on Jira, SNOW, Message on Teams, Slack, Phone Call/Text the Engineer also.

Essentially sending/alerting to Engineer
"""
def escalate_to_engineer(summary: str) -> str:

    print(f"Tool Called - escalate_to_engineer : Esclating to Engineer - Details of Issue [{summary}].")

    # In a real scenario, this would send a Slack message or PagerDuty alert or other ways also.
    esclation_details = {
        "status": "escalated",
        "ticket_id": "INC-999",
        "assigned_to": "On-Call Engineer"
    }

    esclation_details = json.dumps(esclation_details)

    if DEBUG:
      print(f'DEBUG - Tool escalate_to_engineer - Details of Esclation [{esclation_details}]')

    return esclation_details

## Map the Availabe functions - these are the tools available for our Agent to work with.

In [ ]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "get_recent_logs": get_recent_logs,
    "restart_server": restart_server,
    "escalate_to_engineer": escalate_to_engineer,
}

## Set up the Tools Schema for the Agent

In [2]:
tools_schema = [

{
    "type": "function",
    "function":
    {
        "name": "get_server_health",
        "description": "Checks the current CPU and memory usage of a specific server.",
        "parameters":
        {
            "type": "object",
            "properties":
                {
                "server_id":
                    {
                    "type": "string",
                    "description": "The ID of the server, e.g., 'payment-server-01'"
                    }
                },
            "required": ["server_id"]
        }
    }
},

{
    "type": "function",
    "function":
    {
        "name": "fetch_recent_logs",
        "description": "Retrieves the most recent log entries from a server to diagnose errors.",
        "parameters":
        {
            "type": "object",
            "properties":
            {
                "server_id":
                {
                "type": "string",
                "description": "The ID of the server."
                },
                "lines":
                {
                "type": "integer",
                "description": "Number of log lines to fetch."
                }
            },
            "required": ["server_id"]
        }
    }
},


{
    "type": "function",
    "function":
    {
        "name": "restart_service",
        "description": "Restarts a specific server service. Use this when CPU usage is critically high or the process is unresponsive.",
        "parameters":
        {
            "type": "object",
            "properties":
            {
                "server_id":
                {
                "type": "string",
                "description": "The ID of the server to restart."
                }
            },
            "required": ["server_id"]
        }
    }
},


{
    "type": "function",
    "function":
    {
        "name": "escalate_to_engineer",
        "description": "Escalates the issue to a human engineer.Use this when automated fixes fail, or when the error logs indicate a complex issue like a payment gateway failure.",
        "parameters":
        {
            "type": "object",
            "properties":
            {
                "summary":
                {
                "type": "string",
                "description": "A brief summary of the findings (health status, log errors) and why you are escalating."
                }
            },
            "required": ["summary"]
        }
    }
}

]